# Example 8.4 / 1: Built-in and Declared Suffixes on the Steel Model

Companion notebook for **8.4 Suffixes** in the AMPL lab Documentation, section "Built-in and Declared Suffixes on the Steel Model". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Built-in and Declared Suffixes on the Steel Model

This example declares the three suffixes above on the steel model, assigns values with `let`,
solves, and displays declared and built-in suffixes. In `data.py` and `run.py`, `ampl` is an
`AMPL` object that has already read `model.mod`; `ampl.eval(...)` sends the `let` statements,
which amplpy has no method for, and `ampl.display(...)` prints in AMPL's format:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # products
param rate {PROD} > 0;                    # tons produced per hour
param avail >= 0;                         # hours available
param profit {PROD};                      # profit per ton
param market {PROD} >= 0;                 # most tons that can be sold
var Make {p in PROD} >= 0, <= market[p];  # tons produced, up to the market limit
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # total profit
# production hours fit the hours available
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;
suffix priority IN;               # sent to the solver
suffix quality OUT;               # returned by the solver, if it knows the name
suffix audit_code integer LOCAL;  # kept in AMPL only

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.eval("let Make['bands'].priority := 10; let Make['coils'].audit_code := 7;")
ampl.solve(solver="highs", verbose=False)
ampl.display("Make.priority", "Make.quality", "Make.audit_code")
ampl.display("Make", "Make.lb", "Make.ub", "Make.rc")
ampl.display("Time.body", "Time.ub", "Time.slack", "Time.dual")

:     Make.priority Make.quality Make.audit_code    :=
bands       10            0              0
coils        0            0              7
plate        0            0              0
;

:       Make Make.lb Make.ub   Make.rc     :=
bands   6000     0     6000     1.8
coils      0     0     4000    -3.14286
plate   1600     0     3500     0
;

Time.body = 40
Time.ub = 40
Time.slack = 0
Time.dual = 4640



### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := bands 200  coils 140  plate 160;
param profit := bands 25  coils 30  plate 29;
param market := bands 6000  coils 4000  plate 3500;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
let Make['bands'].priority := 10;
let Make['coils'].audit_code := 7;
solve;
display Make.priority, Make.quality, Make.audit_code;
display Make, Make.lb, Make.ub, Make.rc;
display Time.body, Time.ub, Time.slack, Time.dual;

:     Make.priority Make.quality Make.audit_code    :=
bands       10            0              0
coils        0            0              7
plate        0            0              0
;

:       Make Make.lb Make.ub   Make.rc     :=
bands   6000     0     6000     1.8
coils      0     0     4000    -3.14286
plate   1600     0     3500     0
;

Time.body = 40
Time.ub = 40
Time.slack = 0
Time.dual = 4640

